In [6]:
import os
import gzip
import shutil
import tarfile
from pathlib import Path
from tqdm import tqdm

In [2]:
# Defines
ROOT_dir = Path(r"D:\JHAhn\8641_conductivity\8641_CDT-CDT_ex-CDT_in")

In [7]:
gz_files = list(ROOT_dir.rglob("*.gz"))

for gz in tqdm(gz_files, desc="Extracting", unit="file"):
    try:
        if tarfile.is_tarfile(gz):
            with tarfile.open(gz, "r:gz") as tar:
                tar.extractall(path=gz.parent)
        else:  # Single file: '*.gz'
            out_path = gz.with_suffix("")
            with gzip.open(gz, "rb") as fin, open(out_path, "wb") as fout:
                shutil.copyfileobj(fin, fout)
        # gz.unlink()  # ← 원본 삭제를 원하면 주석 해제
    except Exception as e:
        tqdm.write(f"⚠️  {gz} FAILED: {e}")

Extracting: 100%|██████████████████████████████████████████████████████████████████| 822/822 [02:40<00:00,  5.11file/s]


In [10]:
# Destination Folders
dest_dirs = {
    "wConduct": ROOT_dir / "HFC",
    "wsigma_e": ROOT_dir / "EC",
    "wsigma_i": ROOT_dir / "IC",
}
for d in dest_dirs.values():
    d.mkdir(exist_ok=True)

# Move Files
excluded = set(dest_dirs.values())
dups = {}

for f in tqdm(ROOT_dir.rglob("*"), desc="Organising", unit="file"):
    if not f.is_file():
        continue
    if any(p in excluded for p in f.parents):
        continue                     # 이미 목적지 안에 있는 파일은 건너뜀

    name = f.name
    moved = False
    for prefix, dest in dest_dirs.items():
        if name.startswith(prefix):
            target = dest / name
            if target.exists():      # 중복 발생 → 건너뜀
                dups[name] = dups.get(name, 0) + 1
                tqdm.write(f"!!!!! Duplicate skipped: {name}")
            else:
                shutil.move(str(f), str(target))
            moved = True
            break

# Duplication summary
if dups:
    print("\n=== Duplicate summary (skipped) ===")
    for fname, cnt in dups.items():
        print(f"{fname}: {cnt} duplicates")
else:
    print("\nNo duplicates were found.")

Organising: 2378file [00:00, 2446.30file/s]


No duplicates were found.


In [ ]:
dest_dirs = [ROOT_dir / "HFC", ROOT_dir / "EC", ROOT_dir / "IC"]

# 압축 포맷 확장자 리스트
compressed_exts = {".gz", ".tar", ".7z", ".zip"}

for dest in dest_dirs:
    if not dest.exists():
        continue

    for f in tqdm(dest.iterdir(), desc=f"Moving files from {dest.name}", unit="file"):
        # 파일이 아니거나 압축 포맷이면 건너뛴다
        if not f.is_file() or f.suffix.lower() in compressed_exts:
            continue

        target = ROOT_dir / f.name
        if target.exists():
            tqdm.write(f"⚠️ Already exists, skipped: {f.name}")
        else:
            shutil.move(str(f), str(target))

In [11]:
dest_dirs = [ROOT_dir / "HFC", ROOT_dir / "EC", ROOT_dir / "IC"]

for dest in dest_dirs:
    # 폴더가 없으면 건너뛰기
    if not dest.exists():
        continue

    gz_files = list(dest.glob("*.gz"))
    for gz in tqdm(gz_files, desc=f"Moving .gz from {dest.name}", unit="file"):
        try:
            target = ROOT_dir / gz.name
            if target.exists():
                tqdm.write(f"⚠️ Already exists, skipped: {gz.name}")
            else:
                shutil.move(str(gz), str(target))
        except Exception as e:
            tqdm.write(f"❌ Failed to move {gz.name}: {e}")

Moving .gz from IC: 100%|████████████████████████████████████████████████████████| 274/274 [00:00<00:00, 3455.30file/s]


In [16]:
exclude = {"HFC", "EC", "IC"}

folders = [p for p in ROOT_dir.iterdir() if p.is_dir() and p.name not in exclude]

print("Target folders:")
for f in folders:
    print("  ", f.name)

# 4) ROOT_dir에 있는 *.gz 파일 모두 가져오기
gz_files = list(ROOT_dir.glob("*.gz"))

# 5) 파일 이동
for gz in tqdm(gz_files, desc="Moving .gz files", unit="file"):
    moved = False
    for folder in folders:
        # gz 파일명에 folder.name 이 포함되어 있으면 매칭
        if folder.name in gz.name:
            target = folder / gz.name
            if target.exists():
                tqdm.write(f"⚠️ Already exists, skipped: {gz.name}")
            else:
                shutil.move(str(gz), str(target))
            moved = True
            break
    if not moved:
        tqdm.write(f"⚠️ No matching folder for: {gz.name}")

print("\nDone.") 

Target folders:
   ADEPT01_YSJ_NODDI24
   ADEPT02_LKY_NODDI24
   ADEPT03_YKJ_NODDI24
   ADEPT04_JWS_NODDI24
   ADEPT05_CBK_NODDI24
   ADEPT06_PHJ_NODDI24
   ADEPT07_JG_NODDI24
   ADEPT08_KMJ_NODDI24
   ADEPT09_JCJ_NODDI24
   ADEPT10_YPL_NODDI24
   ADEPT16_LJS_NODDI24
   ADEPT18_PST_NODDI24
   ADEPT21_CJM_NODDI24
   ADEPT22_KBS_NODDI24
   ADEPT23_PYS_NODDI24
   ADEPT24_KJH_NODDI24
   ADEPT25_JKR_NODDI24
   ADEPT26_KYJ_NODDI24
   ADEPT27_KKS_NODDI24
   ADEPT28_MSH_NODDI24
   ADEPT29_LBS_NODDI24
   ADEPT30_WJS_NODDI24
   ADEPT31_PTL_NODDI24
   ADEPT32_KSW_NODDI24
   ADEPT33_YCY_NODDI24
   ADEPT34_PDY_NODDI24
   ADEPT36_JJS_NODDI24
   ADEPT37_KTY_NODDI24
   ADEPT38_KWW_NODDI24
   ADEPT39_KKS_NODDI24
   ADEPT40_JOS_NODDI24
   ADEPT41_YBO_NODDI24
   ADEPT42_PSJ_NODDI24
   ADEPT43_SSY_NODDI24
   ADEPT44_WDS_NODDI24
   ADEPT45_KKJ_NODDI24
   ADEPT46_LJB_NODDI24
   ADEPT47_SJH_NODDI24
   ADEPT48_CJK_NODDI24
   ADEPT49_LKJ_NODDI24
   ADEPT50_LJK_NODDI24
   ADEPT51_HBH_NODDI24
   ADEPT52_YCW_NODD

Moving .gz files: 100%|███████████████████████████████████████████████████████████| 822/822 [00:01<00:00, 797.81file/s]


Done.
